# Lab 8.5 &mdash; Parallel Fan-Out

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Fan a ticket from START to several nodes that run in parallel
- Merge their findings with a reducer and fan in to a join node
- Survive a branch that fails -- fault tolerance in a fan-out

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Parallel — fan-out for coverage & speed](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-08-05")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
In the **parallel** (fan-out) shape several agents work the **same input** at once and their outputs are
combined (deck slide 10). In LangGraph you fan out by adding edges from **`START` to several nodes** &mdash;
they run in **one superstep, concurrently** &mdash; and fan in by pointing them all at a **join** node. The
**reducer** (`Annotated[list, add]`) merges their findings no matter the finish order. Two rules the framework
makes easy: keep each result **tagged** with its agent, and make branches **fault-tolerant** &mdash; a node
that *raises* aborts the whole graph, so a down branch must **catch its own error** and return a marker.

In [ ]:
# Three branches on the SAME ticket: billing/tech are real specialists; policy is DOWN.
print("fan-out: START -> billing, tech, policy (parallel) -> join")

## Build it
Complete the **reducer** on `findings`, the down-branch **error marker** in `policy_node`, and the
**fan-out / fan-in edges**. `billing` and `tech` are real specialist nodes.

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain.agents import create_agent

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

@tool
def lookup_invoice(order_id: str) -> str:
    """Look up the charges on an order by its id, e.g. '4471'. Use for billing / charge / refund questions."""
    charges = INVOICES.get(order_id.strip(), [])
    return str(charges) if charges else "no charges found for that order"

@tool
def known_issues(symptom: str) -> str:
    """Look up a known technical issue by symptom keyword, e.g. 'crash' or 'login'. Use for tech problems."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v["bug"] + ": " + v["fix"]
    return "no known issue matched"

class FanState(TypedDict):
    message: str
    findings: Annotated[list, ___]   # TODO: the reducer that MERGES the parallel branches (hint: operator.add)
    summary: str

def build_specialist(tools, role):
    return create_agent(llm, tools, system_prompt=f"You are the {role} specialist. Use ONLY your own tools; answer in one sentence.")

def specialist_node(role, tools):
    agent = build_specialist(tools, role)
    def node(state):
        r = agent.invoke({"messages": [("user", state["message"])]}, config={"recursion_limit": 8})
        return {"findings": [f"{role}: " + r["messages"][-1].content]}
    return node

def policy_node(state):
    """A third branch that is DOWN. It must CATCH its own error -- a raising node aborts the whole graph."""
    try:
        raise RuntimeError("policy service unavailable")
    except Exception as e:
        return ___   # TODO: return an ERROR-marker finding (include type(e).__name__) so the fan-out survives

def join_node(state):
    """Fan-in: combine whatever the branches produced into one summary."""
    return {"summary": " | ".join(sorted(state["findings"]))}

def build_fanout():
    g = StateGraph(FanState)
    g.add_node("billing", specialist_node("billing", [lookup_invoice]))
    g.add_node("tech", specialist_node("tech", [known_issues]))
    g.add_node("policy", policy_node)
    g.add_node("join", join_node)
    for n in ("billing", "tech", "policy"):
        ___   # TODO: fan OUT from START to each branch, and fan each branch IN to "join"
    g.add_edge("join", END)
    return g.compile()

try:
    # Offline sanity (no model call): the down branch degrades to a marker, and the fan-out plan.
    print("down branch ->", policy_node({"message": "x"}))
    print("plan: START -> {billing, tech, policy} in parallel -> join -> END")
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Build the graph and fan one ticket out. Watch the reducer merge findings from branches that finished in any order, and the down branch degrade to a marker.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. Multi-agent runs make several model calls &mdash; keep live runs light on the free tier._

In [ ]:
if not groq_ready():
    print("GROQ_API_KEY not set -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        fan = build_fanout()
        print("graph nodes:", sorted(set(fan.get_graph().nodes) - {"__start__", "__end__"}))

        final = fan.invoke(
            {"message": "I was charged twice on 4471 and the app keeps crashing on login.", "findings": [], "summary": ""},
            config={"recursion_limit": 12})
        print("\nfindings (merged by the reducer):")
        for f in sorted(final["findings"]):
            print("  •", f)
        print("\njoined summary:", final["summary"][:300])
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Three branches run from `START` in **one superstep** &mdash; real parallel execution; latency is the **slowest** branch, not the sum.
- The **reducer** merges all three findings regardless of finish order; the down `policy` branch caught its error and returned a **marker**, so the graph survived.
- You now hold **several** findings and need **one** &mdash; the `join` node starts that; vote/synthesis (Labs 8.7&ndash;8.9) finish it.

## Your turn (open task &mdash; no grader)
Add a fourth branch &mdash; a `billing_review` node that returns a *contradicting* verdict
(e.g. `"billing_review: no refund due"`). **What good looks like:** all four run in parallel, the reducer merges
them (the down branch still degrades to a marker), and you now have a genuine **conflict** to resolve with a vote
(Lab 8.7).

---
### Key takeaway
Fan-out is edges from START to several nodes running in one superstep; a reducer merges their findings and a join fans them back in. A down branch that catches its own error keeps the team alive -- and now you have several outputs to converge.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>